# 09 — Spring Web

**What you'll learn:**

- The Spring MVC request flow: `DispatcherServlet`, handler mapping, response writing
- `@RestController` vs `@Controller` and when to use each
- HTTP method mappings — `@GetMapping`, `@PostMapping`, `@PutMapping`, `@DeleteMapping`, `@PatchMapping`
- Extracting request data — `@PathVariable`, `@RequestParam`, `@RequestBody`, `@RequestHeader`
- Returning responses — return-value mapping, `ResponseEntity`, `@ResponseStatus`
- Content negotiation and automatic JSON (de)serialisation via Jackson
- **Validation** with `@Valid`, `@NotBlank`, `@Size`, `@Email`, `@Positive` and the JSR-380 bean validation set
- **Centralised error handling** with `@ControllerAdvice` and `@ExceptionHandler`
- Returning RFC 7807 `ProblemDetail` for error responses
- Calling other services — **`RestClient`** (synchronous, modern) and **`WebClient`** (reactive)
- **Spring Security essentials** — the `SecurityFilterChain` bean, authentication, authorisation
- **Method security** with `@PreAuthorize` and `@PostAuthorize`
- CORS in one paragraph

Spring Web is where Spring Core turns into a network-facing service. Almost every Spring application you'll ever encounter exposes its functionality over HTTP, and this notebook covers the patterns those services use.

## How a Spring MVC request flows

Every incoming HTTP request lands in the same front controller, the **`DispatcherServlet`**, which Spring Boot wires up automatically. From there:

```text
   HTTP request
        |
        v
   DispatcherServlet
        |
        +-- HandlerMapping       picks @Controller method by URL + verb
        |
        +-- HandlerAdapter       resolves arguments (PathVariable, RequestBody, ...)
        |
        +-- your @Controller     runs business logic, returns a value
        |
        +-- HandlerInterceptor   optional pre/post hooks
        |
        +-- ResponseBodyAdvice   optional response shaping
        |
        v
   HTTP response (status + headers + body, body via HttpMessageConverter)
```

You almost never touch the inner pieces directly — Spring Boot's auto-configuration assembles a working pipeline at startup. Your job is to write `@Controller` classes; the framework wires them into the flow.

## `@RestController` vs `@Controller`

Two stereotypes, two purposes:

- **`@Controller`** — for server-side views. Method return values are *view names* (Thymeleaf templates, JSPs). Used in classic web apps.
- **`@RestController`** — for REST APIs. Method return values are *the response body itself*, serialised to JSON.

`@RestController` is `@Controller` + `@ResponseBody` applied to every method. For modern services that talk JSON, this is the only one you'll use.

In [ ]:
import org.springframework.web.bind.annotation.*;

@RestController
@RequestMapping("/api/users")            // base path for every method in the class
public class UserController {

    private final UserService userService;

    public UserController(UserService userService) {
        this.userService = userService;
    }

    @GetMapping("/{id}")
    public User getUser(@PathVariable long id) {
        return userService.findById(id);   // returned as JSON automatically
    }

    @GetMapping
    public List<User> listUsers(@RequestParam(defaultValue = "0") int page,
                                 @RequestParam(defaultValue = "20") int size) {
        return userService.list(page, size);
    }

    @PostMapping
    public User createUser(@RequestBody NewUser body) {
        return userService.create(body);
    }

    @PutMapping("/{id}")
    public User updateUser(@PathVariable long id, @RequestBody UpdateUser body) {
        return userService.update(id, body);
    }

    @DeleteMapping("/{id}")
    public void deleteUser(@PathVariable long id) {
        userService.delete(id);
    }
}

The five `@GetMapping` / `@PostMapping` / `@PutMapping` / `@DeleteMapping` / `@PatchMapping` annotations are shortcuts for `@RequestMapping(method = ...)`. Use the shortcuts; they read better.

## Extracting request data

Each parameter-level annotation pulls a different piece of the request and maps it to a method parameter.

| Annotation | Source |
|---|---|
| `@PathVariable` | a templated segment in the URL — `/users/{id}` |
| `@RequestParam` | a query string parameter — `?page=0&size=20` |
| `@RequestBody` | the request body, deserialised from JSON |
| `@RequestHeader` | an HTTP header value |
| `@CookieValue` | a cookie value |
| `HttpServletRequest` (no annotation) | the raw request, if you need full control |

In [ ]:
@GetMapping("/{id}/posts/{postId}")
public Post getPost(
    @PathVariable long id,
    @PathVariable long postId,
    @RequestParam(required = false) String filter,
    @RequestHeader("X-Request-Id") String requestId
) {
    // call URL: /api/users/42/posts/7?filter=public  with header X-Request-Id: abc-123
    return postService.find(id, postId, filter);
}

// records are perfect for request bodies — Jackson deserialises by component name
public record NewUser(String email, String name, int age) {}

@PostMapping
public User createUser(@RequestBody NewUser body) {
    return userService.create(body);
}

**JSON conversion is automatic.** Spring Boot bundles Jackson, which serialises any return value to JSON and deserialises every `@RequestBody` from JSON. Records are the ideal shape for request and response DTOs — Jackson handles them by component name, no setters required.

## Returning responses

Three ways to control what comes back, in order of how often you'll use each:

- **Just return the value** — Spring sets `200 OK`, serialises to JSON, done.
- **`@ResponseStatus(...)`** on the method — change the default status (e.g. `201 Created` for POST).
- **`ResponseEntity<T>`** — full control over status, headers, and body in one return value.

In [ ]:
// default — 200 OK with JSON body
@GetMapping("/{id}")
public User getUser(@PathVariable long id) {
    return userService.findById(id);
}

// 201 Created via @ResponseStatus
@PostMapping
@ResponseStatus(HttpStatus.CREATED)
public User createUser(@RequestBody NewUser body) {
    return userService.create(body);
}

// full control — ResponseEntity
@GetMapping("/{id}/avatar")
public ResponseEntity<byte[]> getAvatar(@PathVariable long id) {
    var avatar = userService.findAvatar(id);
    return ResponseEntity.ok()
        .contentType(MediaType.IMAGE_PNG)
        .cacheControl(CacheControl.maxAge(Duration.ofDays(7)))
        .body(avatar);
}

// 204 No Content
@DeleteMapping("/{id}")
@ResponseStatus(HttpStatus.NO_CONTENT)
public void deleteUser(@PathVariable long id) {
    userService.delete(id);
}

## Validation with `@Valid`

Don't accept whatever the client sends. Validate at the controller boundary so bad input never reaches your service layer. Spring integrates with **Jakarta Bean Validation** (JSR-380) — annotate the DTO once and add `@Valid` to the parameter.

In [ ]:
import jakarta.validation.constraints.*;
import jakarta.validation.Valid;

public record NewUser(
    @NotBlank @Email
    String email,

    @NotBlank @Size(min = 2, max = 100)
    String name,

    @Positive @Max(150)
    int age
) {}

@PostMapping
@ResponseStatus(HttpStatus.CREATED)
public User createUser(@Valid @RequestBody NewUser body) {
    return userService.create(body);
}

// @Valid also works on @RequestParam-bound and @PathVariable parameters
@GetMapping
public Page<User> list(
    @RequestParam @Min(0) int page,
    @RequestParam @Min(1) @Max(100) int size
) { /* ... */ }

Common constraints worth knowing: `@NotNull`, `@NotBlank` (non-null, non-empty, non-whitespace), `@NotEmpty` (non-null, length/size > 0), `@Size(min, max)`, `@Min`/`@Max`, `@Positive`/`@Negative`, `@Email`, `@Pattern(regexp = ...)`. When validation fails, Spring throws `MethodArgumentNotValidException`, which by default produces a `400 Bad Request`. We'll handle it cleanly in the next section.

## Centralised error handling

Without help, every exception thrown out of a controller becomes a generic `500 Internal Server Error`. You want the opposite: domain exceptions map to specific HTTP statuses with a clean response body. The mechanism is a single `@ControllerAdvice` (or `@RestControllerAdvice` for JSON-returning APIs) class with `@ExceptionHandler` methods.

In [ ]:
import org.springframework.web.bind.annotation.*;
import org.springframework.http.*;

@RestControllerAdvice
public class GlobalExceptionHandler {

    private static final Logger log = LoggerFactory.getLogger(GlobalExceptionHandler.class);

    // domain exception — map to 404
    @ExceptionHandler(UserNotFoundException.class)
    public ProblemDetail handleNotFound(UserNotFoundException ex) {
        return ProblemDetail.forStatusAndDetail(HttpStatus.NOT_FOUND, ex.getMessage());
    }

    // validation failures — collect field errors into the response
    @ExceptionHandler(MethodArgumentNotValidException.class)
    public ProblemDetail handleValidation(MethodArgumentNotValidException ex) {
        var problem = ProblemDetail.forStatus(HttpStatus.BAD_REQUEST);
        problem.setTitle("Validation failed");
        var errors = ex.getBindingResult().getFieldErrors().stream()
            .collect(Collectors.toMap(FieldError::getField, FieldError::getDefaultMessage));
        problem.setProperty("errors", errors);
        return problem;
    }

    // catch-all — log everything; never expose internals
    @ExceptionHandler(Exception.class)
    public ProblemDetail handleUnexpected(Exception ex) {
        log.error("unexpected error", ex);
        return ProblemDetail.forStatusAndDetail(
            HttpStatus.INTERNAL_SERVER_ERROR, "Internal error");
    }
}

**`ProblemDetail`** implements RFC 7807, the standard JSON shape for HTTP error responses. The default body has `type`, `title`, `status`, `detail`, and `instance` fields; you can add custom fields with `setProperty`. Using `ProblemDetail` makes your error responses self-describing and interoperable with any client that understands the spec.

A single advice class catches exceptions thrown from any controller in the application — Spring picks the most specific `@ExceptionHandler` for each thrown type.

## Calling other services — `RestClient`

For HTTP calls out of your service, Spring Framework 6.1 (Boot 3.2) introduced **`RestClient`** — a fluent, synchronous client that replaces the older `RestTemplate`. Use it for typical service-to-service calls.

In [ ]:
import org.springframework.web.client.RestClient;

@Configuration
public class HttpConfig {
    @Bean
    public RestClient userClient(RestClient.Builder builder) {
        return builder
            .baseUrl("https://api.example.com")
            .defaultHeader("X-Service", "checkout")
            .build();
    }
}

@Service
public class UserGateway {
    private final RestClient client;
    public UserGateway(RestClient userClient) { this.client = userClient; }

    public User fetch(long id) {
        return client.get()
            .uri("/users/{id}", id)
            .retrieve()
            .body(User.class);
    }

    public User create(NewUser user) {
        return client.post()
            .uri("/users")
            .body(user)
            .retrieve()
            .body(User.class);
    }

    public List<User> search(String q) {
        return client.get()
            .uri(b -> b.path("/users").queryParam("q", q).build())
            .retrieve()
            .body(new ParameterizedTypeReference<List<User>>() {});
    }
}

`RestClient` is the new default. It's synchronous and looks like normal blocking code, which is exactly right when you're running on virtual threads (notebook 06) — you get high concurrency without the cognitive load of reactive chains.

**`WebClient`** is the reactive cousin. Same fluent builder, returns `Mono<T>` / `Flux<T>` instead of plain values, and integrates with Project Reactor for non-blocking streams. Use `WebClient` only if you've committed to a reactive stack (WebFlux). For everything else, `RestClient` is the right answer.

## Spring Security — the essentials

Adding `spring-boot-starter-security` to your dependencies gives every endpoint HTTP Basic authentication by default — secure by default, even if you don't write any security configuration. To shape that behaviour, define a `SecurityFilterChain` bean.

In [ ]:
import org.springframework.security.config.annotation.web.builders.HttpSecurity;
import org.springframework.security.web.SecurityFilterChain;

@Configuration
@EnableWebSecurity
public class SecurityConfig {

    @Bean
    public SecurityFilterChain api(HttpSecurity http) throws Exception {
        return http
            .csrf(csrf -> csrf.disable())                  // typical for stateless JSON APIs
            .authorizeHttpRequests(auth -> auth
                .requestMatchers("/api/public/**").permitAll()
                .requestMatchers(HttpMethod.GET, "/api/users/**").hasRole("USER")
                .requestMatchers("/api/admin/**").hasRole("ADMIN")
                .anyRequest().authenticated()
            )
            .httpBasic(withDefaults())                     // or .oauth2ResourceServer(...) for JWT
            .build();
    }

    @Bean
    public PasswordEncoder passwordEncoder() {
        return new BCryptPasswordEncoder();                // never store plain passwords
    }
}

The fluent builder pattern reads like a security policy. `authorizeHttpRequests` lets you list rules in priority order — first match wins. `httpBasic(withDefaults())` enables HTTP Basic; swap for `.formLogin(...)` for browser apps or `.oauth2ResourceServer(...)` for JWT-protected APIs.

A few essential pieces fit around the chain:

- **`UserDetailsService`** — Spring's lookup mechanism for usernames. You implement it (or use one of the built-in implementations) to load credentials from your database.
- **`PasswordEncoder`** — never store plain-text passwords. `BCryptPasswordEncoder` is the standard.
- **`AuthenticationManager`** — Spring uses it internally; you rarely reference it directly.

## Method security

Per-endpoint rules in the filter chain handle coarse-grained authorisation. For fine-grained checks — "only the owner of this account can delete it" — use **method security** annotations directly on your service or controller methods.

In [ ]:
import org.springframework.security.access.prepost.*;

@Configuration
@EnableMethodSecurity                                 // enables @PreAuthorize / @PostAuthorize
public class MethodSecurityConfig {}

@Service
public class AccountService {

    @PreAuthorize("hasRole('ADMIN')")
    public void deleteAll() { /* ... */ }

    // SpEL — refer to method arguments and to the authenticated principal
    @PreAuthorize("hasRole('USER') and #ownerId == authentication.name")
    public Account transfer(@P("ownerId") String ownerId, long amount) { /* ... */ }

    @PostAuthorize("returnObject.owner == authentication.name")
    public Account findById(long id) { /* ... */ }
}

`@PreAuthorize` runs before the method — denied calls never execute. `@PostAuthorize` runs after — useful when the check depends on the returned object. The expressions use **Spring Expression Language (SpEL)**, which lets you reference arguments (with `#name` or `#p0`), the authenticated user (`authentication`, `authentication.name`), and the return value (`returnObject` in `@PostAuthorize`).

## CORS in one paragraph

If your frontend lives at a different origin (e.g. browser app at `app.example.com` calling API at `api.example.com`), the browser enforces **Cross-Origin Resource Sharing** rules. Spring lets you configure CORS globally with a `WebMvcConfigurer`, per-controller or per-method with `@CrossOrigin`, or inside the security chain with `.cors(cors -> cors.configurationSource(...))`. For most cases, a single global config is the right move — list allowed origins, methods, and headers, and apply it across the API.

## What's next

You can now expose a service over HTTP with the right shapes: validated requests, type-safe responses, RFC-compliant errors, outbound calls to other services, and authenticated, authorised access. Combined with the IoC and DI patterns from notebook 08, this is the bulk of what a Spring microservice actually does at runtime.

Notebook 10 plugs in the data layer with **Spring Data** — JPA entity mapping, repository interfaces that write their own SQL, derived query methods, `@Query` for the cases derivation can't reach, paging and sorting, transaction boundaries with `@Transactional`, and database migrations with Flyway. After that, notebook 11 brings Spring Boot's production concerns — auto-configuration, Actuator, testing, messaging with Kafka — and notebook 12 finishes with Spring Cloud.